# E-Commerce Order Analytics — SQL Analysis & Reporting


## Setup

In [1]:
import sqlite3
import os
from datetime import datetime, timedelta

import pandas as pd

os.makedirs("reports", exist_ok=True)
db_path = 'data/ecommerce.db'

---
## Load Data into SQLite

In [2]:
conn = sqlite3.connect(db_path)

for table in ("orders", "order_items", "products", "customers"):
    df = pd.read_csv(f"data/cleaned/{table}.csv")
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"loaded {table}: {len(df)} rows")

print("\ndatabase ready at", db_path)

loaded orders: 600 rows
loaded order_items: 1500 rows
loaded products: 80 rows
loaded customers: 200 rows

database ready at data/ecommerce.db


Helper to run queries and show results nicely:

In [3]:
def run_query(name, sql, show=15):
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    df = pd.read_sql_query(sql, conn)
    if len(df) == 0:
        print("  (no results)")
    else:
        print(df.head(show).to_string(index=False))
        if len(df) > show:
            print(f"  ... ({len(df)} total rows)")
    df.to_csv(f"reports/{name}.csv", index=False)
    return df

---
## Part 3: SQL Analysis

### Basic Queries

**Q1: Total revenue per category**

In [4]:
q1 = run_query("01_revenue_per_category", """
SELECT p.category,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE oi.quantity > 0
GROUP BY p.category
ORDER BY total_revenue DESC
""")


──────────────────────────────────────────────────
  01_revenue_per_category
──────────────────────────────────────────────────
   category  total_revenue
      Books      494515.07
       Home      473731.28
Electronics      438883.36
   Clothing      319962.38


**Q2: Top 10 customers by total order value**

In [5]:
q2 = run_query("02_top10_customers", """
SELECT c.customer_id, c.customer_name, c.customer_type,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.quantity > 0
GROUP BY c.customer_id
ORDER BY total_value DESC
LIMIT 10
""")


──────────────────────────────────────────────────
  02_top10_customers
──────────────────────────────────────────────────
customer_id   customer_name customer_type  total_value
  CUST-0071    Jeffrey Wood           VIP     30393.06
  CUST-0190     Jeremy West       REGULAR     28815.73
  CUST-0131 Ashley Delacruz       REGULAR     26486.40
  CUST-0127    Jack Pollard           VIP     25894.02
  CUST-0158       Eric Hill           VIP     24153.98
  CUST-0105  Andrew Shaw Md       REGULAR     23471.05
  CUST-0010  Thomas Bradley           VIP     23027.87
  CUST-0161   Robert Monroe       PREMIUM     22469.99
  CUST-0041   Justin Baxter       REGULAR     22234.32
  CUST-0198   Curtis Taylor       PREMIUM     21860.11


**Q3: Month-wise order count (last 12 months)**

In [6]:
q3 = run_query("03_monthly_orders", """
SELECT STRFTIME('%Y-%m', order_date) AS month,
       COUNT(DISTINCT order_id) AS order_count
FROM orders
WHERE order_date >= DATE('now', '-12 months')
GROUP BY month
ORDER BY month
""")


──────────────────────────────────────────────────
  03_monthly_orders
──────────────────────────────────────────────────
  (no results)


### Intermediate Queries

**Q4: Customers who placed orders but never had anything delivered**

In [7]:
q4 = run_query("04_never_delivered", """
SELECT DISTINCT c.customer_id, c.customer_name
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE c.customer_id NOT IN (
    SELECT DISTINCT customer_id FROM orders WHERE status = 'DELIVERED'
)
AND c.customer_id != 'UNKNOWN'
""")


──────────────────────────────────────────────────
  04_never_delivered
──────────────────────────────────────────────────
customer_id     customer_name
  CUST-0002    Javier Johnson
  CUST-0003 Kimberly Robinson
  CUST-0004  Daniel Gallagher
  CUST-0005    Monica Herrera
  CUST-0006        Laura Bush
  CUST-0007      Jamie Chavez
  CUST-0008       Maria Lynch
  CUST-0013     James Ferrell
  CUST-0014 Patricia Peterson
  CUST-0015    Jonathan White
  CUST-0016      James Martin
  CUST-0017     Sandra Parker
  CUST-0018 Mr. Philip Cannon
  CUST-0020   Joseph Martinez
  CUST-0021    Benjamin Welch
  ... (112 total rows)


**Q5: Products with more returns than purchases**

In [8]:
q5 = run_query("05_more_returns", """
SELECT p.product_id, p.product_name,
       SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased,
       SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returned
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id
HAVING returned > purchased
""")


──────────────────────────────────────────────────
  05_more_returns
──────────────────────────────────────────────────
  (no results)


**Q6: Return rate per category**

In [9]:
q6 = run_query("06_return_rate", """
SELECT p.category,
       SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returned,
       SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased,
       ROUND(CAST(SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS REAL) /
             NULLIF(SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END), 0) * 100, 2) AS return_rate_pct
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate_pct DESC
""")


──────────────────────────────────────────────────
  06_return_rate
──────────────────────────────────────────────────
   category  returned  purchased  return_rate_pct
   Clothing        53        797             6.65
      Books        60       1169             5.13
       Home        56       1275             4.39
Electronics        21       1096             1.92


### Advanced Queries (Window Functions, CTEs)

**Q7: Running total of revenue per region**

In [10]:
q7 = run_query("07_running_total", """
SELECT o.region_code,
       DATE(o.order_date) AS order_day,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS daily_revenue,
       ROUND(SUM(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)))
           OVER (PARTITION BY o.region_code ORDER BY DATE(o.order_date)
                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.quantity > 0
GROUP BY o.region_code, order_day
ORDER BY o.region_code, order_day
""")


──────────────────────────────────────────────────
  07_running_total
──────────────────────────────────────────────────
region_code  order_day  daily_revenue  running_total
       APAC 2023-07-10        6266.61        6266.61
       APAC 2023-07-25        8911.25       15177.86
       APAC 2023-08-02        3039.25       18217.11
       APAC 2023-08-13        1792.98       20010.09
       APAC 2023-08-21        4032.93       24043.01
       APAC 2023-09-05        4604.49       28647.50
       APAC 2023-09-07        6862.10       35509.60
       APAC 2023-09-09        3526.80       39036.40
       APAC 2023-09-24        1244.07       40280.48
       APAC 2023-10-14        6613.90       46894.38
       APAC 2023-10-21        6498.30       53392.68
       APAC 2023-10-25        7266.30       60658.98
       APAC 2023-10-27        1186.91       61845.89
       APAC 2023-10-30         218.93       62064.82
       APAC 2023-11-08         961.57       63026.39
  ... (509 total rows)


**Q8: Rank products by revenue within each category (DENSE_RANK)**

In [11]:
q8 = run_query("08_product_rank", """
SELECT p.category, p.product_name,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_revenue,
       DENSE_RANK() OVER (
           PARTITION BY p.category
           ORDER BY SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) DESC
       ) AS rank_in_category
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE oi.quantity > 0
GROUP BY p.category, p.product_name
ORDER BY p.category, rank_in_category
""")


──────────────────────────────────────────────────
  08_product_rank
──────────────────────────────────────────────────
category               product_name  total_revenue  rank_in_category
   Books       Deluxe Academic Week       33515.70                 1
   Books       Premium Academic Put       30895.30                 2
   Books Modern Academic Throughout       28738.30                 3
   Books   Compact Academic Develop       26740.30                 4
   Books     Compact Fiction Theory       26558.78                 5
   Books   Compact Self-Help Reason       26345.98                 6
   Books  Elite Non-Fiction Capital       26031.87                 7
   Books        Deluxe Academic Air       25724.00                 8
   Books          Pro Fiction Spend       24817.23                 9
   Books    Basic Non-Fiction Carry       24012.67                10
   Books      Elite Self-Help Claim       24006.13                11
   Books      Modern Comics Trouble       23195.22 

**Q9: Days between consecutive orders per customer (LAG)**

In [12]:
q9 = run_query("09_order_gaps", """
WITH customer_orders AS (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date
    FROM orders
    WHERE customer_id != 'UNKNOWN'
),
gaps AS (
    SELECT customer_id, order_date, prev_order_date,
           CAST(JULIANDAY(order_date) - JULIANDAY(prev_order_date) AS INTEGER) AS days_gap
    FROM customer_orders
    WHERE prev_order_date IS NOT NULL
)
SELECT customer_id, order_date, prev_order_date, days_gap,
       CASE WHEN AVG(days_gap) OVER (PARTITION BY customer_id) > 30
            THEN 'At Risk' ELSE 'Active'
       END AS risk_flag
FROM gaps
ORDER BY customer_id, order_date
""")


──────────────────────────────────────────────────
  09_order_gaps
──────────────────────────────────────────────────
customer_id          order_date     prev_order_date  days_gap risk_flag
  CUST-0001 2024-06-08 14:13:16 2024-04-04 14:36:34        64   At Risk
  CUST-0001 2024-11-21 16:38:11 2024-06-08 14:13:16       166   At Risk
  CUST-0001 2025-03-17 05:16:14 2024-11-21 16:38:11       115   At Risk
  CUST-0004 2025-02-22 01:31:14 2023-12-09 11:24:40       440   At Risk
  CUST-0005 2024-03-29 11:59:14 2024-03-23 23:10:20         5    Active
  CUST-0006 2025-02-25 09:54:18 2025-01-02 08:57:55        54   At Risk
  CUST-0007 2025-02-17 06:43:09 2023-11-11 14:09:03       463   At Risk
  CUST-0008 2024-06-18 09:27:35 2023-09-21 01:04:54       271   At Risk
  CUST-0010 2024-08-10 03:14:04 2024-03-12 01:40:19       151   At Risk
  CUST-0010 2024-10-23 22:29:37 2024-08-10 03:14:04        74   At Risk
  CUST-0010 2025-03-01 08:12:19 2024-10-23 22:29:37       128   At Risk
  CUST-0010 2025-

**Q10: Multi-level CTE — customer segments per month**

In [13]:
q10 = run_query("10_customer_segments", """
WITH monthly_rev AS (
    SELECT o.customer_id, STRFTIME('%Y-%m', o.order_date) AS month,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE oi.quantity > 0 AND o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id, month
),
categorized AS (
    SELECT *, CASE
        WHEN revenue > 10000 THEN 'High'
        WHEN revenue BETWEEN 5000 AND 10000 THEN 'Medium'
        ELSE 'Low'
    END AS segment
    FROM monthly_rev
)
SELECT month, segment, COUNT(DISTINCT customer_id) AS customer_count
FROM categorized
GROUP BY month, segment
ORDER BY month, segment
""")


──────────────────────────────────────────────────
  10_customer_segments
──────────────────────────────────────────────────
  month segment  customer_count
2023-07     Low              18
2023-07  Medium               4
2023-08     Low              17
2023-08  Medium               2
2023-09     Low              21
2023-09  Medium               6
2023-10    High               1
2023-10     Low              13
2023-10  Medium               4
2023-11    High               1
2023-11     Low              18
2023-11  Medium               4
2023-12     Low              21
2023-12  Medium               4
2024-01    High               2
  ... (56 total rows)


**Q11: Customer quartiles by lifetime value (NTILE)**

In [14]:
q11 = run_query("11_customer_quartiles", """
WITH lifetime AS (
    SELECT o.customer_id,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE oi.quantity > 0 AND o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
)
SELECT customer_id, total_value,
       NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
       CASE NTILE(4) OVER (ORDER BY total_value DESC)
           WHEN 1 THEN 'Platinum' WHEN 2 THEN 'Gold'
           WHEN 3 THEN 'Silver' WHEN 4 THEN 'Bronze'
       END AS quartile_label
FROM lifetime
ORDER BY total_value DESC
""")


──────────────────────────────────────────────────
  11_customer_quartiles
──────────────────────────────────────────────────
customer_id  total_value  quartile quartile_label
  CUST-0071     30393.06         1       Platinum
  CUST-0190     28815.73         1       Platinum
  CUST-0131     26486.40         1       Platinum
  CUST-0127     25894.02         1       Platinum
  CUST-0158     24153.98         1       Platinum
  CUST-0105     23471.05         1       Platinum
  CUST-0010     23027.87         1       Platinum
  CUST-0161     22469.99         1       Platinum
  CUST-0041     22234.32         1       Platinum
  CUST-0198     21860.11         1       Platinum
  CUST-0126     21294.06         1       Platinum
  CUST-0168     20827.49         1       Platinum
  CUST-0085     19589.09         1       Platinum
  CUST-0083     19487.35         1       Platinum
  CUST-0015     19234.52         1       Platinum
  ... (182 total rows)


**Q12: Year-over-Year revenue comparison**

In [15]:
q12 = run_query("12_yoy_comparison", """
WITH monthly AS (
    SELECT CAST(STRFTIME('%Y', o.order_date) AS INTEGER) AS year,
           CAST(STRFTIME('%m', o.order_date) AS INTEGER) AS month,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE oi.quantity > 0
    GROUP BY year, month
)
SELECT m.year, m.month, m.revenue, p.revenue AS prev_year_revenue,
       CASE WHEN p.revenue IS NULL OR p.revenue = 0 THEN NULL
            ELSE ROUND((m.revenue - p.revenue) / p.revenue * 100, 2)
       END AS yoy_growth_pct
FROM monthly m
LEFT JOIN monthly p ON m.year = p.year + 1 AND m.month = p.month
ORDER BY m.year, m.month
""")


──────────────────────────────────────────────────
  12_yoy_comparison
──────────────────────────────────────────────────
 year  month   revenue  prev_year_revenue  yoy_growth_pct
 2023      7  63276.26                NaN             NaN
 2023      8  59616.68                NaN             NaN
 2023      9  86337.49                NaN             NaN
 2023     10  64723.33                NaN             NaN
 2023     11  84241.81                NaN             NaN
 2023     12  69571.27                NaN             NaN
 2024      1  80526.74                NaN             NaN
 2024      2  47582.05                NaN             NaN
 2024      3  80759.96                NaN             NaN
 2024      4  45892.11                NaN             NaN
 2024      5 100337.84                NaN             NaN
 2024      6  59852.81                NaN             NaN
 2024      7  72032.09           63276.26           13.84
 2024      8  79865.08           59616.68           33.96
 2024  

**Q13: First and last purchased category per customer**

In [16]:
q13 = run_query("13_category_shift", """
WITH ordered AS (
    SELECT o.customer_id, p.category, o.order_date,
           FIRST_VALUE(p.category) OVER (
               PARTITION BY o.customer_id ORDER BY o.order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
           ) AS first_cat,
           LAST_VALUE(p.category) OVER (
               PARTITION BY o.customer_id ORDER BY o.order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
           ) AS last_cat
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.customer_id != 'UNKNOWN' AND oi.quantity > 0
)
SELECT DISTINCT customer_id, first_cat, last_cat,
       CASE WHEN first_cat != last_cat THEN 'Yes' ELSE 'No' END AS category_shift
FROM ordered
ORDER BY customer_id
""")


──────────────────────────────────────────────────
  13_category_shift
──────────────────────────────────────────────────
customer_id   first_cat    last_cat category_shift
  CUST-0001        Home    Clothing            Yes
  CUST-0002    Clothing       Books            Yes
  CUST-0003 Electronics       Books            Yes
  CUST-0004 Electronics    Clothing            Yes
  CUST-0005        Home       Books            Yes
  CUST-0006       Books       Books             No
  CUST-0007        Home    Clothing            Yes
  CUST-0008    Clothing Electronics            Yes
  CUST-0010        Home    Clothing            Yes
  CUST-0011        Home        Home             No
  CUST-0012 Electronics    Clothing            Yes
  CUST-0013 Electronics        Home            Yes
  CUST-0014        Home        Home             No
  CUST-0015        Home    Clothing            Yes
  CUST-0016       Books    Clothing            Yes
  ... (182 total rows)


**Q14: Cumulative revenue distribution**

In [17]:
q14 = run_query("14_cumulative_dist", """
WITH customer_rev AS (
    SELECT o.customer_id,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE oi.quantity > 0 AND o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
),
total AS (SELECT SUM(revenue) AS grand_total FROM customer_rev)
SELECT cr.customer_id, cr.revenue,
       ROUND(SUM(cr.revenue) OVER (ORDER BY cr.revenue DESC
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS cumulative_revenue,
       ROUND(SUM(cr.revenue) OVER (ORDER BY cr.revenue DESC
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) /
             (SELECT grand_total FROM total) * 100, 2) AS cumulative_pct
FROM customer_rev cr
ORDER BY cr.revenue DESC
""")


──────────────────────────────────────────────────
  14_cumulative_dist
──────────────────────────────────────────────────
customer_id  revenue  cumulative_revenue  cumulative_pct
  CUST-0071 30393.06            30393.06            1.82
  CUST-0190 28815.73            59208.79            3.54
  CUST-0131 26486.40            85695.19            5.12
  CUST-0127 25894.02           111589.21            6.67
  CUST-0158 24153.98           135743.19            8.12
  CUST-0105 23471.05           159214.24            9.52
  CUST-0010 23027.87           182242.11           10.90
  CUST-0161 22469.99           204712.10           12.24
  CUST-0041 22234.32           226946.42           13.57
  CUST-0198 21860.11           248806.53           14.88
  CUST-0126 21294.06           270100.59           16.15
  CUST-0168 20827.49           290928.08           17.39
  CUST-0085 19589.09           310517.17           18.57
  CUST-0083 19487.35           330004.52           19.73
  CUST-0015 19234.52 

**Q15: Cohort retention analysis**

In [18]:
q15 = run_query("15_cohort_retention", """
WITH cohorts AS (
    SELECT customer_id, STRFTIME('%Y-%m', registration_date) AS cohort_month
    FROM customers WHERE customer_id != 'UNKNOWN'
),
order_months AS (
    SELECT customer_id, STRFTIME('%Y-%m', order_date) AS order_month
    FROM orders WHERE customer_id != 'UNKNOWN'
),
activity AS (
    SELECT c.cohort_month, om.order_month,
           (CAST(STRFTIME('%Y', om.order_month || '-01') AS INTEGER) -
            CAST(STRFTIME('%Y', c.cohort_month || '-01') AS INTEGER)) * 12 +
           (CAST(STRFTIME('%m', om.order_month || '-01') AS INTEGER) -
            CAST(STRFTIME('%m', c.cohort_month || '-01') AS INTEGER)) AS months_since,
           COUNT(DISTINCT c.customer_id) AS active_customers
    FROM cohorts c
    JOIN order_months om ON c.customer_id = om.customer_id
    GROUP BY c.cohort_month, om.order_month
),
sizes AS (
    SELECT cohort_month, COUNT(*) AS cohort_size FROM cohorts GROUP BY cohort_month
)
SELECT a.cohort_month, a.months_since, s.cohort_size, a.active_customers,
       ROUND(CAST(a.active_customers AS REAL) / s.cohort_size * 100, 2) AS retention_rate
FROM activity a
JOIN sizes s ON a.cohort_month = s.cohort_month
WHERE a.months_since BETWEEN 0 AND 6
ORDER BY a.cohort_month, a.months_since
""")


──────────────────────────────────────────────────
  15_cohort_retention
──────────────────────────────────────────────────
cohort_month  months_since  cohort_size  active_customers  retention_rate
     2023-08             1            4                 1           25.00
     2023-08             3            4                 1           25.00
     2023-08             4            4                 1           25.00
     2023-08             6            4                 1           25.00
     2023-09             2            4                 2           50.00
     2023-09             3            4                 1           25.00
     2023-09             6            4                 1           25.00
     2023-10             1            6                 2           33.33
     2023-10             2            6                 1           16.67
     2023-10             3            6                 1           16.67
     2023-11             0            9                 1    

**Q16: Products frequently bought together**

In [19]:
q16 = run_query("16_bought_together", """
SELECT p1.product_name AS product_a, p2.product_name AS product_b,
       COUNT(*) AS times_together
FROM order_items oi1
JOIN order_items oi2 ON oi1.order_id = oi2.order_id AND oi1.product_id < oi2.product_id
JOIN products p1 ON oi1.product_id = p1.product_id
JOIN products p2 ON oi2.product_id = p2.product_id
WHERE oi1.quantity > 0 AND oi2.quantity > 0
GROUP BY oi1.product_id, oi2.product_id
HAVING times_together >= 2
ORDER BY times_together DESC
LIMIT 20
""")


──────────────────────────────────────────────────
  16_bought_together
──────────────────────────────────────────────────
                product_a                  product_b  times_together
    Compact Headphones Ok  Elite Non-Fiction Capital               5
    Compact Headphones Ok   Compact Self-Help Reason               4
      Elite Decor Special    Elite Lighting Probably               4
       Pro Furniture Lead      Modern T-Shirts Worry               4
Modern Smartphones Around          Modern Decor Half               4
Modern Smartphones Around Modern Academic Throughout               4
    Elite Self-Help Claim     Basic Dresses Thousand               4
    Modern T-Shirts Worry       Deluxe Academic Week               4
 Compact Self-Help Reason    Elite Lighting Probably               4
 Classic Dresses Response       Pro Fiction Security               3
  Compact Lighting Nation       Pro Dresses Congress               3
      Basic Furniture Guy     Basic Dresses Thou

---
## Part 4: Command-Line Reporting Tool

A simple report generator that takes a report type and date range, then shows key metrics with period-over-period comparison. Uses only sqlite3 — no external SQL libraries.

In [ ]:
def generate_report(report_type, start_date, end_date):
    cur = conn.cursor()
    cur.row_factory = sqlite3.Row

    cur.execute("""
        SELECT COUNT(DISTINCT o.order_id) AS total_orders,
               COUNT(DISTINCT o.customer_id) AS unique_customers,
               COALESCE(ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2), 0) AS revenue
        FROM orders o
        LEFT JOIN order_items oi ON o.order_id = oi.order_id AND oi.quantity > 0
        WHERE DATE(o.order_date) BETWEEN ? AND ?
    """, (start_date, end_date))
    row = cur.fetchone()
    total_orders = row[0] or 0
    revenue = row[2] or 0
    customers = row[1] or 0

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    delta = end_dt - start_dt
    prev_end = start_dt - timedelta(days=1)
    prev_start = prev_end - delta

    cur.execute("""
        SELECT COUNT(DISTINCT o.order_id), COUNT(DISTINCT o.customer_id),
               COALESCE(ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2), 0)
        FROM orders o
        LEFT JOIN order_items oi ON o.order_id = oi.order_id AND oi.quantity > 0
        WHERE DATE(o.order_date) BETWEEN ? AND ?
    """, (prev_start.strftime("%Y-%m-%d"), prev_end.strftime("%Y-%m-%d")))
    prev = cur.fetchone()
    prev_orders, prev_customers, prev_revenue = (prev[0] or 0), (prev[1] or 0), (prev[2] or 0)

    def pct(curr, prev):
        if prev == 0: return "N/A"
        ch = ((curr - prev) / prev) * 100
        return f"{'↑' if ch >= 0 else '↓'} {abs(ch):.1f}%"

    cur.execute("""
        SELECT p.product_name,
               ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS rev
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE DATE(o.order_date) BETWEEN ? AND ? AND oi.quantity > 0
        GROUP BY p.product_id ORDER BY rev DESC LIMIT 3
    """, (start_date, end_date))
    top_prods = cur.fetchall()

    print(f"\n{'='*55}")
    print(f"  {report_type.upper()} REPORT: {start_date} → {end_date}")
    print(f"{'='*55}")
    print(f"  Orders    : {total_orders:>8,}   {pct(total_orders, prev_orders)}")
    print(f"  Revenue   : ${revenue:>10,.2f}   {pct(revenue, prev_revenue)}")
    print(f"  Customers : {customers:>8,}   {pct(customers, prev_customers)}")
    print(f"{'─'*55}")
    print("  Top 3 Products:")
    for i, p in enumerate(top_prods, 1):
        print(f"    {i}. {p[0][:35]:35s} ${p[1]:,.2f}")
    print(f"{'='*55}")

Let's test it with a monthly report:

In [21]:
generate_report("monthly", "2024-01-01", "2025-06-30")


  MONTHLY REPORT: 2024-01-01 → 2025-06-30
  Orders    :      435   ↑ 163.6%
  Revenue   : $1,299,325.26   ↑ 203.7%
  Customers :      173   ↑ 53.1%
───────────────────────────────────────────────────────
  Top 3 Products:
    1. Modern Decor Half                   $27,974.99
    2. Basic Dresses Part                  $27,679.93
    3. Lite Headphones Court               $27,010.74


In [22]:
generate_report("monthly", "2024-06-01", "2024-12-31")


  MONTHLY REPORT: 2024-06-01 → 2024-12-31
  Orders    :      156   ↓ 16.1%
  Revenue   : $493,120.35   ↓ 3.8%
  Customers :       98   ↓ 24.6%
───────────────────────────────────────────────────────
  Top 3 Products:
    1. Modern Decor Half                   $18,691.89
    2. Deluxe Academic Week                $17,201.21
    3. Basic Dresses Part                  $13,985.00


---
## Part 5: Edge Case Tests

Testing what happens with problematic data

In [ ]:
results = []

def test(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    results.append((name, status, detail))
    icon = "✓" if condition else "✗"
    print(f"  {icon} [{status}] {name}")
    if detail:
        print(f"           {detail}")

In [24]:
cur = conn.cursor()

# Test 1: orphan order items
cur.execute("""
    SELECT COUNT(*) FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""")
orphans = cur.fetchone()[0]
test("Orphan order_items", orphans == 0, f"found {orphans} orphans")

# Test 2: discount > 100
cur.execute("SELECT COUNT(*) FROM order_items WHERE discount_percent > 100")
over100 = cur.fetchone()[0]
test("Discount > 100% capped", over100 == 0, f"found {over100} uncapped")

# Test 3: negative revenue from valid sales
cur.execute("""
    SELECT COUNT(*) FROM order_items
    WHERE quantity > 0 AND quantity * unit_price * (1 - discount_percent/100.0) < 0
""")
neg_rev = cur.fetchone()[0]
test("No negative revenue from positive qty", neg_rev == 0, f"found {neg_rev}")

# Test 4: zero quantity
cur.execute("""
    SELECT COALESCE(SUM(quantity * unit_price * (1 - discount_percent/100.0)), 0)
    FROM order_items WHERE quantity = 0
""")
zero_rev = cur.fetchone()[0]
test("Zero qty items generate zero revenue", zero_rev == 0, f"revenue = {zero_rev}")

# Test 5: future dates don't break things
cur.execute("SELECT COUNT(*) FROM orders WHERE DATE(order_date) > DATE('now')")
future = cur.fetchone()[0]
test("Future dates handled", True, f"found {future} future orders")

# Test 6: NULL customer_ids cleaned
cur.execute("SELECT COUNT(*) FROM orders WHERE customer_id IS NULL OR customer_id = '' OR customer_id = 'NULL'")
raw_nulls = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM orders WHERE customer_id = 'UNKNOWN'")
unknowns = cur.fetchone()[0]
test("NULL customer_ids → UNKNOWN", raw_nulls == 0, f"nulls: {raw_nulls}, unknowns: {unknowns}")

# Test 7: negative qty flagged as returns
cur.execute("SELECT COUNT(*) FROM order_items WHERE quantity < 0 AND is_return = 1")
flagged = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM order_items WHERE quantity < 0")
total_neg = cur.fetchone()[0]
test("Negative qty flagged as returns", flagged == total_neg, f"flagged: {flagged}/{total_neg}")

# Test 8: numeric types intact
cur.execute("SELECT AVG(unit_price), AVG(quantity) FROM order_items")
avgs = cur.fetchone()
ok = all(isinstance(v, (int, float)) for v in avgs if v is not None)
test("Numeric columns intact", ok, f"avg price={avgs[0]:.2f}, avg qty={avgs[1]:.2f}")

print(f"\n{'─'*50}")
passed = sum(1 for _, s, _ in results if s == "PASS")
print(f"  Total: {len(results)}  |  Passed: {passed}  |  Failed: {len(results)-passed}")

  ✓ [PASS] Orphan order_items
           found 0 orphans
  ✓ [PASS] Discount > 100% capped
           found 0 uncapped
  ✓ [PASS] No negative revenue from positive qty
           found 0
  ✓ [PASS] Zero qty items generate zero revenue
           revenue = 0
  ✓ [PASS] Future dates handled
           found 0 future orders
  ✓ [PASS] NULL customer_ids → UNKNOWN
           nulls: 0, unknowns: 20
  ✓ [PASS] Negative qty flagged as returns
           flagged: 57/57
  ✓ [PASS] Numeric columns intact
           avg price=503.00, avg qty=2.76

──────────────────────────────────────────────────
  Total: 8  |  Passed: 8  |  Failed: 0


## Cleanup

In [25]:
conn.close()
print("database connection closed")
print("all done!")

database connection closed
all done!
